# 16 — A UI that does not own agent logic
Streamlit sends validated requests to FastAPI; the application and Flow remain
on the server. The Streamlit app lives in workshop/ui.py.



**Advanced follow-up to lesson 13 (project).**


## Problem

Learners need a human-facing view without duplicating the harness.


## Concept

Streamlit is a thin HTTP client for the FastAPI boundary.

Streamlit owns presentation and request construction only. The server remains the authority for model execution and action policy.


## Where this fits in the whole workshop

```text
USER · CLI · API
   │
   ▼
 13 PROJECT        run_assistant(): memory in → Flow → report out   ◀ YOU ARE HERE
   │
   ├─ 06 MEMORY          recall this customer's saved constraint
   ├─ 09 CONTROL         validate the question before any model call
   ├─ 08 FLOW            typed state · route · one bounded retry
   │    └─ 07 CREW            Researcher → Analyst → Critic
   │         └─ 02 AGENT LOOP      model decides → tool runs → observation → answer
   │              ├─ 01 MODEL          the LLM call itself
   │              ├─ 03 TOOLS          typed Python functions the agent may call
   │              ├─ 04 MCP            tools served by a separate process
   │              └─ 05 KNOWLEDGE      cited passages from company documents
   └─ 09 CONTROL         validate the report → human approval → mock email

 around everything:
   10 RESILIENCE      checkpoint and resume
   11 OBSERVABILITY   trace what actually ran
   12 EVALUATION      checks that fail loudly
```

The flagged row is what this lesson adds. Every other row is the same system, taught in its own lesson.


## Architecture of this lesson

```text
Streamlit form → FastAPI → run_assistant → report with cited evidence → Streamlit
```

form → API request → report and source IDs. A tool result or validation failure is an observation, not an instruction the model may bypass.


## Step 1 — Inspect the validated request payload and the command that starts Streamlit

Inspect the validated request payload and the command that starts Streamlit.


In [ ]:
import os
import sys
from pathlib import Path

here = Path(__file__).resolve() if "__file__" in globals() else Path.cwd() / "lesson.py"
sys.path.insert(0, str(next(p for p in here.parents if (p / "pyproject.toml").exists())))

from workshop.api import api
from workshop.ui_contract import decision_payload

print("Payload:", decision_payload("customer-a", "Should Northstar AI pilot Meridian AI?"))
print("UI command: uv run --extra advanced streamlit run workshop/ui.py")
print("API URL:", os.getenv("WORKSHOP_API_URL", "http://localhost:8000"))


## Failure mode to know about

A UI that directly runs CrewAI would bypass authentication, persistence, and approval. Keep it as a client.


## Production implication

Keep credentials out of logs and browser persistence. Show provenance and failure states to reviewers.


## Your turn

Open the UI and compare its payload to the API schema.

**Checkpoint:** The UI displays cited evidence and never executes CrewAI directly.
